In [1]:
import os, sys
os.chdir('../../')
os.environ["DPM_TQDM"] = "False"

GMFLOW = os.path.join("submodules", "GMFlow")
sys.path.insert(0, GMFLOW)

In [2]:
from backbones.gmdit import GMDiT

# 사용 예
model = GMDiT()
print(model)

/home/scpark/miniconda3/envs/dual/lib/python3.11/site-packages/mmcv/cnn/bricks/transformer.py:33: UserWarning: Fail to import ``MultiScaleDeformableAttention`` from ``mmcv.ops.multi_scale_deform_attn``, You should install ``mmcv-full`` if you need this module. 
  warnings.warn('Fail to import ``MultiScaleDeformableAttention`` from '
2025-12-09 06:32:53.208535: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/scpark/logpx_samplers/submodules/GMFlow/lib/ops/gmflow_ops/gmflow_ops.py:31: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

cannot get type annotation for Parameter scheduler of <class 'lib.pipelines.gmdit_pipeline.GMDiTPipeline'>.


Loading pipeline components...:   0%|          | 0/4 [00:00<?, ?it/s]

Expected types for id2label: (typing.Dict[int, str], <class 'NoneType'>), got typing.Dict[str, str].


In [3]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

def display_class_images(pixel_samples, class_ids=None, figsize_per_image=3.5, display_title=""):
    print(display_title)
    samples = list(pixel_samples)
    processed = []
    for img in samples:
        if isinstance(img, Image.Image):
            arr = np.array(img)
        else:
            arr = np.array(img)
            if arr.ndim == 3 and arr.shape[0] in (1,3):
                arr = arr.transpose(1, 2, 0)
        processed.append(arr)

    n = len(processed)
    if class_ids is None:
        class_ids = list(range(n))

    fig, axes = plt.subplots(1, n, figsize=(n * figsize_per_image, figsize_per_image))
    if n == 1:
        axes = [axes]

    # 전체 제목 추가
    # if display_title:
    #     fig.suptitle(display_title, fontsize=10, y=1.1)  # y 값을 1.02로 올려서 제목을 위로 이동
    # 플롯 레이아웃 조정: top을 낮춰서 제목과 플롯 사이에 공간 확보
    
    for ax, img_arr, lbl in zip(axes, processed, class_ids):
        ax.imshow(img_arr)
        ax.axis("off")
        #ax.set_title(f"Class {lbl}", fontsize=8)

    plt.show()


In [4]:
!ls logs/gmdit/mobile_ll6_5k/

s9_g0.0_tx-2.0_te-2.0  s9_g1.0_tx-2.0_te-2.0


In [ ]:
import torch
from utils.util import get_pt

class_ids = [i for i in range(1, 201, 20)]
device = 'cuda:0'

noise_schedule = model.get_noise_schedule()
model_fn = model.get_model_fn(noise_schedule, pos_conds=class_ids, guidance_scale=1.5)
noises = model.get_noise(seeds=class_ids)

NFE = 9

device = 'cuda:0'
from solvers.taylor.solver.gdual_solver import GDual_Solver
from solvers.taylor.transform.loglinear_transform6 import LogLinearTransform
from solvers.taylor.extractor.table_extractor import Extractor

for gamma in (0.0, 1.0):
    for tau_x in (-2.0, 2.0):
        for tau_e in (-2.0, 2.0):
            print(gamma, tau_x, tau_e)

            noise_schedule = model.get_noise_schedule()
            extractor = Extractor(steps=NFE)
            transform = LogLinearTransform(gamma_push=True, gamma_init=gamma, tau_x_init=tau_x, tau_e_init=tau_e, eps=1e-2)
            solver = GDual_Solver(
                noise_schedule,
                steps=NFE,
                transform=transform,
                param_extractor=extractor,
                skip_type="time_uniform_flow",
                flow_shift=1.0,
                pred_order=1,
                corr_order=2,
                order1_kappa=True,
                order2_kappa=True,
                use_corrector=True,
                time_learning=True,
                train_mode=True,
                checkpoint=True
            ).to(device)

            with torch.no_grad():
                latents = solver.sample(noises, model_fn)['samples']
                samples = model.decode_vae(latents, raw_output=False, pil_output=True)
            display_class_images(samples['pil_output'], class_ids, display_title='GDual ViT.')

            pt_file = get_pt(f'logs/gmdit/mobile_ll6_5k/s9_g{gamma}_tx{tau_x}_te{tau_e}', 'latest')
            if pt_file is None:
                continue
            
            print(pt_file)
            state_dict = torch.load(pt_file, map_location='cpu', weights_only=False)['solver_state_dict']
            solver.load_state_dict(state_dict, strict=True)

            with torch.no_grad():
                latents = solver.sample(noises, model_fn)['samples']
                samples = model.decode_vae(latents, raw_output=False, pil_output=True)
            display_class_images(samples['pil_output'], class_ids, display_title='GDual ViT.')

            print()
            print()
            print()
            print()



0.0 -2.0 -2.0


/home/scpark/miniconda3/envs/dual/lib/python3.11/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


In [ ]:
print('done')

done
